In [ ]:
# PRACTICAL NO. 6 - LSTM AUTOENCODER

# 1. IMPORT LIBRARIES

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, RepeatVector, TimeDistributed, Dense
from tensorflow.keras.callbacks import EarlyStopping

from google.colab import files

import warnings
warnings.filterwarnings("ignore")

print("TensorFlow Version:", tf.__version__)

# 2. UPLOAD AND LOAD KAGGLE DATASET

print("\nUpload your Kaggle Energy Consumption CSV file:")

uploaded = files.upload()
file_name = list(uploaded.keys())[0]
df = pd.read_csv(file_name)

print("\nDataset Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nFirst 5 Rows:")
print(df.head())

# 3. SELECT TIME AND ENERGY COLUMN

TIME_COL = df.columns[0]

# Automatically select the first numeric column
numeric_columns = df.select_dtypes(include=np.number).columns.tolist()

if len(numeric_columns) == 0:
    raise ValueError("No numeric column found. Please check your dataset.")

VALUE_COL = numeric_columns[0]

print("\nSelected Time Column:", TIME_COL)
print("Selected Energy Column:", VALUE_COL)

# 4. PREPROCESS DATA

df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce")
df[VALUE_COL] = pd.to_numeric(df[VALUE_COL], errors="coerce")

# Remove missing/invalid values
df = df.dropna(subset=[TIME_COL, VALUE_COL])

# Sort according to timestamp
df = df.sort_values(TIME_COL)

# Remove duplicate timestamps
df = df.drop_duplicates(subset=[TIME_COL])

df = df.reset_index(drop=True)

print("\nAfter preprocessing:", df.shape)
print("\nMissing Values:")
print(df[[TIME_COL, VALUE_COL]].isnull().sum())

# 5. VISUALIZE ORIGINAL TIME SERIES

plt.figure(figsize=(15, 5))

plt.plot(df[TIME_COL], df[VALUE_COL], color="blue")

plt.title("Original Energy Consumption Time Series")
plt.xlabel("Time")
plt.ylabel("Energy Consumption")

plt.grid(True)
plt.show()

# 6. NORMALIZE DATA

scaler = MinMaxScaler(feature_range=(0, 1))

energy_values = df[[VALUE_COL]].values
scaled_data = scaler.fit_transform(energy_values)

print("\nOriginal Data:")
print(energy_values[:5])

print("\nNormalized Data:")
print(scaled_data[:5])

# 7. CREATE SEQUENCES

SEQUENCE_LENGTH = 24

def create_sequences(data, sequence_length):
    sequences = []
    for i in range(len(data) - sequence_length + 1):
        sequences.append(data[i:i + sequence_length])
    return np.array(sequences)

X = create_sequences(scaled_data, SEQUENCE_LENGTH)

print("\nSequence Shape:", X.shape)

# 8. TRAIN-TEST SPLIT

train_size = int(len(X) * 0.80)

X_train = X[:train_size]
X_test = X[train_size:]

print("\nTraining Data Shape:", X_train.shape)
print("Testing Data Shape:", X_test.shape)

# 9. BUILD LSTM AUTOENCODER

timesteps = X_train.shape[1]
features = X_train.shape[2]

# Input
inputs = Input(shape=(timesteps, features))

# Encoder
encoded = LSTM(64, activation="tanh", return_sequences=False)(inputs)

# Repeat encoded vector
repeated = RepeatVector(timesteps)(encoded)

# Decoder
decoded = LSTM(64, activation="tanh", return_sequences=True)(repeated)

# Output
outputs = TimeDistributed(Dense(features))(decoded)

# Create model
autoencoder = Model(inputs, outputs)

print("\nLSTM AUTOENCODER MODEL:")
autoencoder.summary()

# 10. COMPILE MODEL

autoencoder.compile(optimizer="adam", loss="mse")

# 11. TRAIN MODEL

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history = autoencoder.fit(
    X_train,
    X_train,
    epochs=50,
    batch_size=32,
    validation_split=0.10,
    shuffle=False,
    callbacks=[early_stopping],
    verbose=1
)

# 12. VISUALIZE TRAINING LOSS

plt.figure(figsize=(10, 5))

plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")

plt.title("LSTM Autoencoder Training and Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")

plt.legend()
plt.grid(True)
plt.show()

# 13. RECONSTRUCT TEST DATA

X_test_pred = autoencoder.predict(X_test, verbose=0)

print("\nOriginal Test Shape:", X_test.shape)
print("Reconstructed Shape:", X_test_pred.shape)

# 14. CALCULATE RECONSTRUCTION ERROR

reconstruction_error = np.mean(
    np.square(X_test - X_test_pred),
    axis=(1, 2)
)

print("\nFirst 10 Reconstruction Errors:")
print(reconstruction_error[:10])

# 15. CALCULATE TRAINING RECONSTRUCTION ERROR

X_train_pred = autoencoder.predict(X_train, verbose=0)

train_reconstruction_error = np.mean(
    np.square(X_train - X_train_pred),
    axis=(1, 2)
)

# 16. CALCULATE ANOMALY THRESHOLD

threshold = (
    np.mean(train_reconstruction_error)
    + 3 * np.std(train_reconstruction_error)
)

anomalies = reconstruction_error > threshold

print("\nAnomaly Threshold:", threshold)
print("Number of Anomalies:", np.sum(anomalies))

# 17. VISUALIZE RECONSTRUCTION ERROR

plt.figure(figsize=(15, 6))

plt.plot(
    reconstruction_error,
    color="blue",
    label="Reconstruction Error"
)

plt.axhline(
    threshold,
    color="red",
    linestyle="--",
    label="Anomaly Threshold"
)

plt.scatter(
    np.where(anomalies)[0],
    reconstruction_error[anomalies],
    color="red",
    s=40,
    label="Anomaly"
)

plt.title("Energy Consumption Anomaly Detection")
plt.xlabel("Test Sequence")
plt.ylabel("Reconstruction Error")

plt.legend()
plt.grid(True)
plt.show()

# 18. ORIGINAL VS RECONSTRUCTED SEQUENCE

sequence_index = 0

original_sequence = scaler.inverse_transform(X_test[sequence_index])
reconstructed_sequence = scaler.inverse_transform(X_test_pred[sequence_index])

plt.figure(figsize=(12, 5))

plt.plot(
    original_sequence,
    label="Original",
    color="blue",
    linewidth=2
)

plt.plot(
    reconstructed_sequence,
    label="Reconstructed",
    color="orange",
    linestyle="--",
    linewidth=2
)

plt.title("Original vs Reconstructed Energy Consumption")
plt.xlabel("Time Step")
plt.ylabel("Energy Consumption")

plt.legend()
plt.grid(True)
plt.show()

# 19. MODEL EVALUATION

mse = mean_squared_error(X_test.flatten(), X_test_pred.flatten())
mae = mean_absolute_error(X_test.flatten(), X_test_pred.flatten())
rmse = np.sqrt(mse)

# 20. RECONSTRUCTION ACCURACY

max_error = np.max(np.abs(X_test - X_test_pred))

if max_error == 0:
    accuracy = 100
else:
    accuracy = (1 - (mae / max_error)) * 100

accuracy = max(0, min(100, accuracy))

# 21. DISPLAY FINAL RESULTS

print("\n===================================")
print("       MODEL EVALUATION")
print("===================================")

print("Mean Squared Error (MSE):", mse)
print("Mean Absolute Error (MAE):", mae)
print("Root Mean Squared Error (RMSE):", rmse)
print("Reconstruction Accuracy:", round(accuracy, 2), "%")

print("\n===================================")
print("           FINAL RESULT")
print("===================================")

print("LSTM Autoencoder trained successfully.")
print("Input Sequence Length:", SEQUENCE_LENGTH)
print("Training Sequences:", len(X_train))
print("Testing Sequences:", len(X_test))
print("Anomalies Detected:", np.sum(anomalies))
print("Accuracy:", round(accuracy, 2), "%")